**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 31: 온톨로지 RAG - Apache AGE 기반 지식 그래프 추론

## 학습 목표

1️⃣ 온톨로지의 개념과 구성 요소를 이해한다
2️⃣ 벡터 RAG, 그래프 RAG, 온톨로지 RAG의 차이를 비교한다
3️⃣ Apache AGE의 특징과 기본 사용법을 학습한다
4️⃣ 도메인 온톨로지를 설계하고 AGE 그래프로 구축한다
5️⃣ 온톨로지 기반 추론과 LLM 연동을 실습한다
6️⃣ 기업 거버넌스/법률/금융 도메인 활용 사례를 살펴본다

---

### 실습 환경
- **GPU**: 선택사항
- **필수 패키지**: psycopg2, networkx, openai, langchain
- **외부 서비스**: PostgreSQL + Apache AGE 확장


In [ ]:
# 💡 setup.sh 실행했으면 이 셀은 건너뛰세요 (참고용 — 본 노트북이 필요로 하는 패키지)
# 패키지 설치
# !pip install -q psycopg2-binary networkx matplotlib openai langchain langchain-openai rdflib

In [ ]:
# 패키지 임포트 및 환경 설정
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional

# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# PostgreSQL + AGE 연결 정보
# DB_CONFIG = {
#     "host": "localhost",
#     "port": 5432,
#     "dbname": "agedb",
#     "user": "postgres",
#     "password": "password"
# }

print("패키지 로드 완료")

---

## 1️⃣ 온톨로지란?

### 개념

온톨로지(Ontology)는 특정 도메인의 **개념, 속성, 관계를 형식적으로 정의**한 지식 표현 체계입니다.  
단순한 지식 그래프와 달리, 온톨로지는 **스키마 레벨의 의미론적 구조**를 제공합니다.

```
지식 그래프:  (이재용) --[CEO]--> (삼성전자)           ← 인스턴스 레벨
온톨로지:     (Person) --[worksAt]--> (Organization)   ← 스키마 레벨
              + 제약조건: Person은 최대 1개의 Organization에 worksAt 가능
              + 추론규칙: worksAt의 역관계는 hasEmployee
```

### 핵심 구성 요소

| 구성 요소 | 설명 | 예시 |
|-----------|------|------|
| **클래스 (Class)** | 개념의 범주 | Person, Organization, Product |
| **프로퍼티 (Property)** | 클래스의 속성 | name, age, founded_year |
| **관계 (Relation)** | 클래스 간 연결 | worksAt, produces, locatedIn |
| **인스턴스 (Instance)** | 클래스의 구체적 개체 | 이재용 (Person), 삼성전자 (Organization) |
| **제약조건 (Constraint)** | 관계의 규칙 | 카디널리티, 도메인/레인지 |

### OWL / RDF 소개

- **RDF (Resource Description Framework)**: 트리플 (주어-술어-목적어) 기반 데이터 모델
- **RDFS**: RDF 스키마, 클래스와 프로퍼티의 계층 구조 정의
- **OWL (Web Ontology Language)**: 더 풍부한 표현력 (동치, 제약, 추론 규칙 등)

```turtle
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:Samsung rdf:type ex:Organization .
ex:Samsung ex:produces ex:Galaxy .
ex:JYLee rdf:type ex:Person .
ex:JYLee ex:worksAt ex:Samsung .
```

In [ ]:
# Python에서 간단한 온톨로지 정의

class OntologySchema:
    """도메인 온톨로지 스키마를 정의하는 클래스"""

    def __init__(self, name: str):
        self.name = name
        self.classes = {}       # 클래스 정의
        self.relations = {}     # 관계 정의
        self.constraints = []   # 제약조건

    def add_class(self, class_name: str, parent: str = None, properties: Dict = None):
        """클래스 정의 추가"""
        self.classes[class_name] = {
            "parent": parent,
            "properties": properties or {}
        }

    def add_relation(self, name: str, domain: str, range_cls: str,
                     cardinality: str = "many-to-many", inverse: str = None):
        """관계 정의 추가"""
        self.relations[name] = {
            "domain": domain,
            "range": range_cls,
            "cardinality": cardinality,
            "inverse": inverse
        }

    def add_constraint(self, description: str):
        """제약조건 추가"""
        self.constraints.append(description)

    def display(self):
        """온톨로지 스키마 출력"""
        print(f"=== 온톨로지: {self.name} ===")
        print(f"\n[클래스] ({len(self.classes)}개)")
        for cls_name, cls_info in self.classes.items():
            parent = f" (상위: {cls_info['parent']})" if cls_info['parent'] else ""
            print(f"  - {cls_name}{parent}")
            for prop, ptype in cls_info['properties'].items():
                print(f"      .{prop}: {ptype}")

        print(f"\n[관계] ({len(self.relations)}개)")
        for rel_name, rel_info in self.relations.items():
            inv = f" (역: {rel_info['inverse']})" if rel_info['inverse'] else ""
            print(f"  - {rel_info['domain']} --[{rel_name}]--> {rel_info['range']}  [{rel_info['cardinality']}]{inv}")

        if self.constraints:
            print(f"\n[제약조건] ({len(self.constraints)}개)")
            for c in self.constraints:
                print(f"  - {c}")


# IT 기업 도메인 온톨로지 정의
ontology = OntologySchema("한국 IT 기업 온톨로지")

# 클래스 정의
ontology.add_class("Entity", properties={"name": "string", "description": "string"})
ontology.add_class("Organization", parent="Entity", properties={"founded_year": "int", "industry": "string"})
ontology.add_class("Company", parent="Organization", properties={"revenue": "float", "employees": "int"})
ontology.add_class("Person", parent="Entity", properties={"birth_year": "int", "role": "string"})
ontology.add_class("Product", parent="Entity", properties={"category": "string", "release_year": "int"})
ontology.add_class("Technology", parent="Entity", properties={"domain": "string"})
ontology.add_class("Location", parent="Entity", properties={"country": "string", "city": "string"})

# 관계 정의
ontology.add_relation("worksAt", "Person", "Organization", "many-to-one", inverse="hasEmployee")
ontology.add_relation("produces", "Company", "Product", "one-to-many", inverse="producedBy")
ontology.add_relation("uses", "Product", "Technology", "many-to-many")
ontology.add_relation("locatedIn", "Organization", "Location", "many-to-one")
ontology.add_relation("competesWith", "Company", "Company", "many-to-many")
ontology.add_relation("subsidiaryOf", "Company", "Company", "many-to-one", inverse="hasSubsidiary")

# 제약조건
ontology.add_constraint("Company는 반드시 하나의 Location에 locatedIn 관계를 가져야 한다")
ontology.add_constraint("Person의 worksAt 관계는 최대 1개의 Organization으로 제한된다")
ontology.add_constraint("competesWith 관계는 대칭적이다 (A competesWith B이면 B competesWith A)")

ontology.display()

---

## 2️⃣ 벡터 RAG vs 그래프 RAG vs 온톨로지 RAG

### 구조적 차이 비교

| 특성 | 벡터 RAG | 그래프 RAG | 온톨로지 RAG |
|------|----------|------------|-------------|
| **데이터 구조** | 벡터 임베딩 | 지식 그래프 (트리플) | 온톨로지 + 지식 그래프 |
| **스키마** | 없음 | 암묵적 | 명시적 (OWL/RDF) |
| **검색 방식** | 코사인 유사도 | 그래프 탐색 | 온톨로지 추론 + 그래프 탐색 |
| **추론 능력** | 제한적 | 경로 기반 추론 | 논리적 추론 (상속, 전이, 대칭) |
| **관계 표현** | 암묵적 | 명시적 (레이블) | 형식적 (도메인/레인지/제약) |
| **장점** | 구축 쉬움, 범용적 | 관계 추론 | 도메인 전문 추론, 일관성 보장 |
| **단점** | 관계 손실 | 스키마 부재 | 구축 비용 높음 |
| **적합 도메인** | 일반 QA | 관계 질문 | 기업 거버넌스, 법률, 금융 등 전문 도메인 |

### 추론 능력 비교 예시

```
질문: "반도체를 생산하는 회사의 CEO가 졸업한 대학은?"

벡터 RAG:
  → 유사한 문서 검색 → 단일 홉 답변만 가능 → 실패 가능성 높음

그래프 RAG:
  → (반도체) ←[생산]← (삼성전자) ←[CEO]← (이재용) →[졸업]→ (서울대)
  → 경로 추적으로 답변 가능

온톨로지 RAG:
  → 온톨로지: produces의 domain은 Company, CEO의 range는 Person
  → 추론: Company.produces(Product) ∧ Person.ceoOf(Company) ∧ Person.graduatedFrom(University)
  → 스키마 기반 정확한 쿼리 생성 + 결과 검증
```


In [ ]:
# 세 가지 RAG 접근법 비교 시각화

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. 벡터 RAG
ax1 = axes[0]
ax1.set_title("Vector RAG", fontsize=14, fontweight="bold")
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
import numpy as np
np.random.seed(42)
x = np.random.uniform(2, 8, 15)
y = np.random.uniform(2, 8, 15)
ax1.scatter(x, y, s=100, c="#4ECDC4", alpha=0.7)
ax1.scatter([5], [5], s=200, c="#FF6B6B", marker="*", zorder=5, label="Query")
for i in range(3):
    ax1.plot([5, x[i]], [5, y[i]], "--", color="gray", alpha=0.5)
ax1.set_xlabel("Embedding Dim 1")
ax1.set_ylabel("Embedding Dim 2")
ax1.legend()

# 2. 그래프 RAG
ax2 = axes[1]
ax2.set_title("Graph RAG", fontsize=14, fontweight="bold")
G2 = nx.DiGraph()
G2.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("A", "E"), ("E", "F"), ("B", "F")])
pos2 = nx.spring_layout(G2, seed=42)
nx.draw(G2, pos2, ax=ax2, with_labels=True, node_color="#4ECDC4",
        node_size=800, font_size=12, arrows=True, arrowsize=15, edge_color="gray")

# 3. 온톨로지 RAG
ax3 = axes[2]
ax3.set_title("Ontology RAG", fontsize=14, fontweight="bold")
G3 = nx.DiGraph()
# 스키마 레벨 (상단)
G3.add_edges_from([("Person", "Org"), ("Org", "Product"), ("Org", "Location")])
# 인스턴스 레벨 (하단)
G3.add_edges_from([("p1", "o1"), ("o1", "pr1"), ("o1", "loc1")])
# 스키마-인스턴스 연결
G3.add_edges_from([("p1", "Person"), ("o1", "Org"), ("pr1", "Product"), ("loc1", "Location")])
pos3 = {
    "Person": (1, 4), "Org": (3, 4), "Product": (5, 4), "Location": (3, 5.5),
    "p1": (1, 1.5), "o1": (3, 1.5), "pr1": (5, 1.5), "loc1": (3, 0)
}
schema_nodes = ["Person", "Org", "Product", "Location"]
instance_nodes = ["p1", "o1", "pr1", "loc1"]
nx.draw_networkx_nodes(G3, pos3, nodelist=schema_nodes, node_color="#FF6B6B", node_size=800, ax=ax3)
nx.draw_networkx_nodes(G3, pos3, nodelist=instance_nodes, node_color="#4ECDC4", node_size=600, ax=ax3)
nx.draw_networkx_labels(G3, pos3, font_size=9, ax=ax3)
nx.draw_networkx_edges(G3, pos3, ax=ax3, arrows=True, arrowsize=12, edge_color="gray")
ax3.text(0.5, 6.2, "Schema Level", fontsize=10, color="#FF6B6B", fontweight="bold")
ax3.text(0.5, 2.5, "Instance Level", fontsize=10, color="#4ECDC4", fontweight="bold")

plt.tight_layout()
plt.savefig("rag_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("세 가지 RAG 방식의 구조적 차이를 시각화했습니다.")

---

## 3️⃣ Apache AGE 소개

### Apache AGE란?

**Apache AGE (A Graph Extension)**는 PostgreSQL의 그래프 데이터베이스 확장입니다.

| 특성 | 설명 |
|------|------|
| **기반** | PostgreSQL 확장 (Extension) |
| **쿼리 언어** | openCypher (Neo4j 호환) |
| **장점** | SQL + Cypher 동시 사용, ACID 트랜잭션 |
| **비용** | 오픈소스 (Apache License 2.0) |
| **성능** | PostgreSQL 최적화 엔진 활용 |

### 왜 Apache AGE인가?

```
Neo4j 대비 Apache AGE의 장점:
  1. PostgreSQL 생태계 활용 (기존 인프라 재사용)
  2. SQL과 Cypher를 함께 사용 가능
  3. 완전한 오픈소스 (Neo4j Enterprise는 유료)
  4. 관계형 데이터 + 그래프 데이터 통합 관리
  5. ACID 트랜잭션 보장
```

### 설치 방법

```bash
# Docker로 Apache AGE 설치 (가장 간단)
docker pull apache/age
docker run -p 5432:5432 -e POSTGRES_PASSWORD=password apache/age

# 또는 PostgreSQL에 직접 설치
# sudo apt-get install postgresql-16-age
```

---

## 4️⃣ Apache AGE 설치 및 기본 사용법

### AGE 초기화 및 그래프 생성

In [ ]:
# Apache AGE 연결 및 관리 클래스

import psycopg2


class ApacheAGE:
    """Apache AGE 그래프 데이터베이스 관리 클래스"""

    def __init__(self, host="localhost", port=5432, dbname="agedb",
                 user="postgres", password="password"):
        self.conn = psycopg2.connect(
            host=host, port=port, dbname=dbname,
            user=user, password=password
        )
        self.conn.autocommit = True
        self.cursor = self.conn.cursor()
        self._init_age()
        print(f"Apache AGE 연결 성공: {host}:{port}/{dbname}")

    def _init_age(self):
        """AGE 확장 로드"""
        self.cursor.execute("CREATE EXTENSION IF NOT EXISTS age;")
        self.cursor.execute("LOAD 'age';")
        self.cursor.execute("SET search_path = ag_catalog, '$user', public;")

    def create_graph(self, graph_name: str):
        """그래프 생성"""
        try:
            self.cursor.execute(f"SELECT create_graph('{graph_name}');")
            print(f"그래프 '{graph_name}' 생성 완료")
        except psycopg2.errors.InvalidParameterValue:
            print(f"그래프 '{graph_name}'이 이미 존재합니다")

    def drop_graph(self, graph_name: str):
        """그래프 삭제"""
        self.cursor.execute(f"SELECT drop_graph('{graph_name}', true);")
        print(f"그래프 '{graph_name}' 삭제 완료")

    def cypher(self, graph_name: str, query: str, params: Dict = None) -> List:
        """Cypher 쿼리 실행"""
        sql = f"SELECT * FROM cypher('{graph_name}', $$ {query} $$) AS (result agtype);"
        self.cursor.execute(sql)
        results = self.cursor.fetchall()
        return results

    def add_node(self, graph_name: str, label: str, properties: Dict):
        """노드 추가"""
        props_str = json.dumps(properties, ensure_ascii=False)
        query = f"CREATE (n:{label} {props_str}) RETURN n"
        return self.cypher(graph_name, query)

    def add_edge(self, graph_name: str, from_label: str, from_props: Dict,
                 rel_type: str, to_label: str, to_props: Dict, rel_props: Dict = None):
        """엣지 추가"""
        rel_props_str = json.dumps(rel_props, ensure_ascii=False) if rel_props else ""
        from_match = json.dumps(from_props, ensure_ascii=False)
        to_match = json.dumps(to_props, ensure_ascii=False)
        query = f"""
        MATCH (a:{from_label} {from_match}), (b:{to_label} {to_match})
        CREATE (a)-[r:{rel_type} {rel_props_str}]->(b)
        RETURN r
        """
        return self.cypher(graph_name, query)

    def query_neighbors(self, graph_name: str, node_name: str, max_hops: int = 2) -> List:
        """이웃 노드 검색"""
        query = f"""
        MATCH path = (n {{name: '{node_name}'}})-[*1..{max_hops}]-(m)
        RETURN path
        """
        return self.cypher(graph_name, query)

    def close(self):
        self.cursor.close()
        self.conn.close()


print("ApacheAGE 클래스 정의 완료")
print()
print("사용 예시:")
print('  age = ApacheAGE(host="localhost", dbname="agedb")')
print('  age.create_graph("my_ontology")')
print('  age.add_node("my_ontology", "Person", {"name": "이재용", "role": "회장"})')

In [ ]:
# Cypher 쿼리 기초 - 주요 패턴 정리

cypher_examples = {
    "노드 생성": """
CREATE (n:Person {name: '이재용', role: '회장'})
RETURN n
""",
    "관계 생성": """
MATCH (a:Person {name: '이재용'}), (b:Company {name: '삼성전자'})
CREATE (a)-[:WORKS_AT {since: 2020}]->(b)
RETURN a, b
""",
    "단순 검색": """
MATCH (p:Person)-[:WORKS_AT]->(c:Company)
WHERE c.name = '삼성전자'
RETURN p.name AS employee, c.name AS company
""",
    "멀티홉 검색": """
MATCH (p:Person)-[:WORKS_AT]->(c:Company)-[:PRODUCES]->(prod:Product)
WHERE prod.category = '반도체'
RETURN p.name, c.name, prod.name
""",
    "경로 탐색": """
MATCH path = shortestPath(
  (a:Person {name: '이재용'})-[*]-(b:Product {name: 'HBM'})
)
RETURN path
""",
    "집계 쿼리": """
MATCH (c:Company)-[:PRODUCES]->(p:Product)
RETURN c.name AS company, count(p) AS product_count
ORDER BY product_count DESC
"""
}

print("=== Cypher 쿼리 기초 ===\n")
for title, query in cypher_examples.items():
    print(f"--- {title} ---")
    print(query)

---

## 5️⃣ 온톨로지 RAG 실습

### 전체 파이프라인

```
┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│  도메인 온톨로지   │ →  │ AGE 그래프 구축   │ →  │ 온톨로지 기반 추론 │ →  │    LLM 연동      │
│  스키마 설계       │    │ 노드/엣지 생성    │    │ 스키마 활용 쿼리   │    │  답변 생성       │
└──────────────────┘    └──────────────────┘    └──────────────────┘    └──────────────────┘
```

In [ ]:
# Step 1: 도메인 온톨로지 설계 (기업 거버넌스 — 한국 대기업 그룹)

corp_ontology = OntologySchema("기업 거버넌스 온톨로지")

# ─── 클래스 정의 ────────────────────────────────────
corp_ontology.add_class("Group",      properties={"name": "string", "founded_year": "int"})
corp_ontology.add_class("Subsidiary", properties={"name": "string", "industry": "string"})
corp_ontology.add_class("Person",     properties={"name": "string", "role": "string"})
corp_ontology.add_class("University", properties={"name": "string", "country": "string"})
corp_ontology.add_class("Industry",   properties={"name": "string", "sector": "string"})

# ─── 관계 정의 ────────────────────────────────────
corp_ontology.add_relation("HAS_CHAIRMAN",        "Group",      "Person",     "one-to-one")
corp_ontology.add_relation("SUBSIDIARY_OF",       "Subsidiary", "Group",      "many-to-one")
corp_ontology.add_relation("HAS_CEO",             "Subsidiary", "Person",     "one-to-one")
corp_ontology.add_relation("GRADUATED_FROM",      "Person",     "University", "many-to-one")
corp_ontology.add_relation("BELONGS_TO_INDUSTRY", "Subsidiary", "Industry",   "many-to-one")
corp_ontology.add_relation("COMPETES_WITH",       "Subsidiary", "Subsidiary", "many-to-many")

# ─── 제약조건 (거버넌스 규칙) ──────────────────────
corp_ontology.add_constraint("한 Group의 회장(HAS_CHAIRMAN)은 정확히 1명이어야 한다 (cardinality = 1)")
corp_ontology.add_constraint("한 Person은 두 개 이상의 Group의 회장이 될 수 없다 (1인 1회장)")
corp_ontology.add_constraint("회장(HAS_CHAIRMAN)과 CEO(HAS_CEO)가 같은 Person이면 거버넌스 충돌")
corp_ontology.add_constraint("COMPETES_WITH 관계는 대칭적이다 (A competes_with B ⇔ B competes_with A)")

corp_ontology.display()


In [ ]:
# Step 2: AGE 그래프 구축 (한국 대기업 거버넌스 데이터 — companies.txt 기반)

corporate_data = {
    "nodes": [
        # 그룹 3개
        {"label": "Group", "props": {"name": "삼성그룹",       "founded_year": 1938}},
        {"label": "Group", "props": {"name": "SK그룹",         "founded_year": 1953}},
        {"label": "Group", "props": {"name": "현대자동차그룹", "founded_year": 1947}},
        # 회장 3명
        {"label": "Person", "props": {"name": "이재용", "role": "회장"}},
        {"label": "Person", "props": {"name": "최태원", "role": "회장"}},
        {"label": "Person", "props": {"name": "정의선", "role": "회장"}},
        # 계열사 9개
        {"label": "Subsidiary", "props": {"name": "삼성전자",     "industry": "반도체/전자"}},
        {"label": "Subsidiary", "props": {"name": "삼성물산",     "industry": "건설/상사"}},
        {"label": "Subsidiary", "props": {"name": "삼성SDS",      "industry": "IT서비스"}},
        {"label": "Subsidiary", "props": {"name": "SK하이닉스",   "industry": "반도체"}},
        {"label": "Subsidiary", "props": {"name": "SK이노베이션", "industry": "에너지"}},
        {"label": "Subsidiary", "props": {"name": "SK텔레콤",     "industry": "통신"}},
        {"label": "Subsidiary", "props": {"name": "현대자동차",   "industry": "자동차"}},
        {"label": "Subsidiary", "props": {"name": "기아",         "industry": "자동차"}},
        {"label": "Subsidiary", "props": {"name": "현대모비스",   "industry": "자동차부품"}},
        # CEO 9명
        {"label": "Person", "props": {"name": "한종희", "role": "CEO"}},
        {"label": "Person", "props": {"name": "오세철", "role": "CEO"}},
        {"label": "Person", "props": {"name": "이준희", "role": "CEO"}},
        {"label": "Person", "props": {"name": "곽노정", "role": "CEO"}},
        {"label": "Person", "props": {"name": "박상규", "role": "CEO"}},
        {"label": "Person", "props": {"name": "유영상", "role": "CEO"}},
        {"label": "Person", "props": {"name": "장재훈", "role": "CEO"}},
        {"label": "Person", "props": {"name": "송호성", "role": "CEO"}},
        {"label": "Person", "props": {"name": "이규석", "role": "CEO"}},
        # 학교 6개
        {"label": "University", "props": {"name": "서울대학교",  "country": "한국"}},
        {"label": "University", "props": {"name": "인하대학교",  "country": "한국"}},
        {"label": "University", "props": {"name": "연세대학교",  "country": "한국"}},
        {"label": "University", "props": {"name": "시카고대학교", "country": "미국"}},
        {"label": "University", "props": {"name": "고려대학교",  "country": "한국"}},
        {"label": "University", "props": {"name": "한양대학교",  "country": "한국"}},
    ],
    "edges": [
        # 그룹 → 회장
        {"from": "삼성그룹",       "rel": "HAS_CHAIRMAN", "to": "이재용"},
        {"from": "SK그룹",         "rel": "HAS_CHAIRMAN", "to": "최태원"},
        {"from": "현대자동차그룹", "rel": "HAS_CHAIRMAN", "to": "정의선"},
        # 계열사 → 그룹
        {"from": "삼성전자",       "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "삼성물산",       "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "삼성SDS",        "rel": "SUBSIDIARY_OF", "to": "삼성그룹"},
        {"from": "SK하이닉스",     "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "SK이노베이션",   "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "SK텔레콤",       "rel": "SUBSIDIARY_OF", "to": "SK그룹"},
        {"from": "현대자동차",     "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        {"from": "기아",           "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        {"from": "현대모비스",     "rel": "SUBSIDIARY_OF", "to": "현대자동차그룹"},
        # 계열사 → CEO
        {"from": "삼성전자",       "rel": "HAS_CEO", "to": "한종희"},
        {"from": "삼성물산",       "rel": "HAS_CEO", "to": "오세철"},
        {"from": "삼성SDS",        "rel": "HAS_CEO", "to": "이준희"},
        {"from": "SK하이닉스",     "rel": "HAS_CEO", "to": "곽노정"},
        {"from": "SK이노베이션",   "rel": "HAS_CEO", "to": "박상규"},
        {"from": "SK텔레콤",       "rel": "HAS_CEO", "to": "유영상"},
        {"from": "현대자동차",     "rel": "HAS_CEO", "to": "장재훈"},
        {"from": "기아",           "rel": "HAS_CEO", "to": "송호성"},
        {"from": "현대모비스",     "rel": "HAS_CEO", "to": "이규석"},
        # 인물 → 학교
        {"from": "이재용", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "최태원", "rel": "GRADUATED_FROM", "to": "시카고대학교"},
        {"from": "정의선", "rel": "GRADUATED_FROM", "to": "고려대학교"},
        {"from": "한종희", "rel": "GRADUATED_FROM", "to": "인하대학교"},
        {"from": "오세철", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "이준희", "rel": "GRADUATED_FROM", "to": "연세대학교"},
        {"from": "곽노정", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "박상규", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "유영상", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "장재훈", "rel": "GRADUATED_FROM", "to": "서울대학교"},
        {"from": "송호성", "rel": "GRADUATED_FROM", "to": "한양대학교"},
        {"from": "이규석", "rel": "GRADUATED_FROM", "to": "서울대학교"},
    ],
}


def build_age_graph(age_client, graph_name: str, data: Dict):
    """Apache AGE 에 온톨로지 기반 그래프를 구축."""
    age_client.create_graph(graph_name)
    for node in data["nodes"]:
        age_client.add_node(graph_name, node["label"], node["props"])
        print(f"  노드: [{node['label']}] {node['props']['name']}")
    for edge in data["edges"]:
        print(f"  관계: {edge['from']} --[{edge['rel']}]--> {edge['to']}")
    print(f"\n그래프 '{graph_name}' 구축 완료")
    print(f"  노드: {len(data['nodes'])}개, 엣지: {len(data['edges'])}개")


# AGE 실행 중일 때:
# age = ApacheAGE(host="localhost", dbname="agedb")
# build_age_graph(age, "corporate_ontology", corporate_data)

print("기업 거버넌스 그래프 데이터 준비 완료")
print(f"  노드: {len(corporate_data['nodes'])}개")
print(f"  엣지: {len(corporate_data['edges'])}개")


In [ ]:
# Step 3: 온톨로지 기반 추론 엔진

class OntologyReasoner:
    """온톨로지 스키마를 활용한 추론 엔진"""

    def __init__(self, ontology: OntologySchema, graph_data: Dict):
        self.ontology = ontology
        self.graph = self._build_internal_graph(graph_data)

    def _build_internal_graph(self, data: Dict) -> nx.DiGraph:
        G = nx.DiGraph()
        for node in data["nodes"]:
            G.add_node(node["props"]["name"], label=node["label"], **node["props"])
        for edge in data["edges"]:
            G.add_edge(edge["from"], edge["to"], relation=edge["rel"])
        return G

    def find_by_class(self, class_name: str) -> List[str]:
        return [n for n, d in self.graph.nodes(data=True) if d.get("label") == class_name]

    def find_related(self, entity: str, relation: str = None, max_hops: int = 2) -> List[Dict]:
        results = []
        visited = set()

        def traverse(node, depth):
            if depth > max_hops or node in visited:
                return
            visited.add(node)
            for _, target, data in self.graph.out_edges(node, data=True):
                if relation is None or data["relation"] == relation:
                    results.append({"from": node, "relation": data["relation"],
                                    "to": target, "depth": depth})
                traverse(target, depth + 1)

        traverse(entity, 1)
        return results

    def check_constraint(self, entity: str, action: str) -> List[str]:
        """제약조건 검증 — 1인 1회장 위반 검사."""
        warnings = []
        if action == "appoint_chairman":
            # entity = Person, 이 사람이 이미 다른 그룹의 회장인가?
            chairman_of = []
            for n, d in self.graph.nodes(data=True):
                if d.get("label") == "Group":
                    for _, tgt, ed in self.graph.out_edges(n, data=True):
                        if ed["relation"] == "HAS_CHAIRMAN" and tgt == entity:
                            chairman_of.append(n)
            if len(chairman_of) > 1:
                warnings.append(
                    f"경고: {entity} 은 이미 {chairman_of} 의 회장 — 1인 1회장 위반"
                )
        return warnings

    def ontology_guided_query(self, question: str) -> Dict:
        """온톨로지 스키마를 활용한 구조화된 쿼리"""
        result = {"question": question, "identified_classes": [],
                  "identified_relations": [], "query_path": [], "results": []}
        class_keywords = {
            "Group":      ["그룹", "재벌"],
            "Subsidiary": ["계열사", "자회사", "회사"],
            "Person":     ["회장", "CEO", "임원", "사람"],
            "University": ["대학", "학교", "졸업"],
        }
        for cls, keywords in class_keywords.items():
            if any(kw in question for kw in keywords):
                result["identified_classes"].append(cls)
        return result


# ── 추론 엔진 테스트 ──────────────────────────
reasoner = OntologyReasoner(corp_ontology, corporate_data)

print("=== 온톨로지 기반 추론 테스트 (기업 거버넌스) ===\n")

print("[Group 클래스 인스턴스]")
for g in reasoner.find_by_class("Group"):
    print(f"  - {g}")

print("\n['삼성그룹' 관련 정보 (2홉)]")
related = reasoner.find_related("삼성그룹", max_hops=2)
for r in related:
    indent = "  " * r["depth"]
    print(f"{indent}{r['from']} --[{r['relation']}]--> {r['to']}")


In [ ]:
# Step 4: 온톨로지 RAG - LLM 연동

from openai import OpenAI


class OntologyRAG:
    """온톨로지 기반 RAG 시스템"""

    def __init__(self, ontology: OntologySchema, reasoner: OntologyReasoner):
        self.ontology = ontology
        self.reasoner = reasoner
        self.client = OpenAI()

    def _build_schema_context(self) -> str:
        ctx = f"도메인: {self.ontology.name}\n\n클래스:\n"
        for cls_name, cls_info in self.ontology.classes.items():
            parent = f" (상위: {cls_info['parent']})" if cls_info['parent'] else ""
            ctx += f"  - {cls_name}{parent}: {cls_info['properties']}\n"
        ctx += "\n관계:\n"
        for rel_name, rel_info in self.ontology.relations.items():
            ctx += f"  - {rel_info['domain']} --[{rel_name}]--> {rel_info['range']}\n"
        ctx += "\n제약조건:\n"
        for c in self.ontology.constraints:
            ctx += f"  - {c}\n"
        return ctx

    def _build_graph_context(self, entities: List[str]) -> str:
        ctx = "관련 지식 그래프 데이터:\n"
        for entity in entities:
            related = self.reasoner.find_related(entity, max_hops=2)
            for r in related:
                ctx += f"  ({r['from']}) --[{r['relation']}]--> ({r['to']})\n"
        return ctx

    def query(self, question: str, entities: List[str]) -> str:
        schema_ctx = self._build_schema_context()
        graph_ctx = self._build_graph_context(entities)
        prompt = f"""당신은 기업 거버넌스 도메인 전문가입니다.
다음 온톨로지 스키마와 지식 그래프 데이터를 참고하여 질문에 정확하게 답변하세요.

=== 온톨로지 스키마 ===
{schema_ctx}

=== 지식 그래프 데이터 ===
{graph_ctx}

질문: {question}

온톨로지의 클래스와 관계 정의를 활용하여 체계적으로 답변하세요.
제약조건(1인 1회장 등)을 반드시 확인하고, 위반 사항이 있으면 경고하세요."""

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content


# 사용 예시
# rag = OntologyRAG(corp_ontology, reasoner)
# answer = rag.query("이재용이 회장인 그룹의 계열사 CEO들이 어느 학교를 나왔는지 알려줘", ["이재용"])
# print(answer)

print("OntologyRAG 클래스 정의 완료")
print()
print("온톨로지 RAG 동작 흐름:")
print("  1. 질문 분석 → 관련 엔티티 추출 (예: '이재용')")
print("  2. 온톨로지 스키마 컨텍스트 생성")
print("  3. 그래프 탐색으로 관련 데이터 수집 (4-hop 회장→그룹→계열사→CEO→학교)")
print("  4. 제약조건 검증 (1인 1회장 등)")
print("  5. LLM 에 스키마 + 데이터 전달 → 답변 생성")


In [ ]:
# 온톨로지 기반 추론 시나리오: 회장 추가 임명 시 거버넌스 위반 검사

def check_chairman_appointment(reasoner: OntologyReasoner,
                                person: str, target_group: str):
    """특정 인물을 target_group 의 회장으로 새로 임명할 때 거버넌스 위반을 검사."""
    print(f"=== 회장 임명 거버넌스 검사 ===")
    print(f"  대상 인물: {person}")
    print(f"  추가 임명 그룹: {target_group}")
    print()

    # 1) 현재 회장 직책 확인 (Person 은 in-edge HAS_CHAIRMAN 으로 연결됨)
    current_chairman_of = []
    for source, _, data in reasoner.graph.in_edges(person, data=True):
        if data["relation"] == "HAS_CHAIRMAN":
            current_chairman_of.append(source)

    print("  [현재 회장 직책]")
    if current_chairman_of:
        for g in current_chairman_of:
            print(f"    • {g} ─ HAS_CHAIRMAN ─→ {person}")
    else:
        print("    (없음)")
    print()

    # 2) target_group 의 현 회장 확인
    target_current_chairman = None
    for _, tgt, data in reasoner.graph.out_edges(target_group, data=True):
        if data["relation"] == "HAS_CHAIRMAN":
            target_current_chairman = tgt
            break

    print(f"  [{target_group} 의 현 회장]")
    print(f"    {target_current_chairman or '(없음)'}")
    print()

    # 3) 거버넌스 위반 판정
    warnings = []
    if person in current_chairman_of:
        warnings.append(
            f"⚠ {person} 은 이미 {current_chairman_of[0]} 의 회장입니다 — 1인 1회장 위반"
        )
    if current_chairman_of and target_group not in current_chairman_of:
        warnings.append(
            f"⚠ {person} 의 새 회장직 추가 → 1인 1회장 제약 위반"
        )
    if target_current_chairman and target_current_chairman != person:
        warnings.append(
            f"⚠ {target_group} 은 이미 {target_current_chairman} 이 회장 — Group HAS_CHAIRMAN cardinality(1) 위반"
        )

    print("  [거버넌스 위반]")
    if warnings:
        for w in warnings:
            print(f"    {w}")
    else:
        print("    위반 없음 — 임명 가능")


# 테스트 — 이재용을 SK그룹 회장으로 추가 임명 시도
check_chairman_appointment(reasoner, "이재용", "SK그룹")


---

## 5.5️⃣ 세 가지 RAG 비교 — 같은 질문, 다른 답

지금까지 만든 OntologyRAG 가 정말 가치가 있는지 확인하기 위해 **Naive RAG · Graph RAG · Ontology RAG** 세 방식을 같은 질문으로 비교합니다.

| 항목 | Naive RAG | Graph RAG | Ontology RAG |
|---|---|---|---|
| 인덱싱 단위 | 텍스트 청크 | 노드 + 엣지 | 스키마 + 노드 + 엣지 + 제약 |
| 검색 방식 | 임베딩 유사도 (Top-K) | 시작 노드부터 N-hop BFS | 스키마 기반 경로 + 제약 검사 |
| 클래스/관계 의미 | ✗ (문자로만) | △ (이름만) | ✓ (도메인·치역·제약) |
| 제약조건 자동 검증 | ✗ | ✗ | ✓ |
| **1인 1회장** 시나리오 | 놓침 | 회장직 사실은 보지만 규칙 모름 | **자동 위반 검출** |

> **시나리오** — *이재용*을 *SK그룹*의 회장으로 추가 임명해도 거버넌스 문제가 없는가?
> (스키마에는 "한 사람은 한 그룹의 회장만" 이라는 1인 1회장 제약이 명시되어 있음)


In [ ]:
# 비교용 데이터 보강 — 산업 분류 + 경쟁사 관계 추가
# (§5.5 RAG 비교는 corporate_data 그대로 사용. §5.7 월드 모델에서 활용)

corporate_data_v2 = {
    "nodes": list(corporate_data["nodes"]) + [
        # Industry 클래스 인스턴스
        {"label": "Industry", "props": {"name": "반도체",    "sector": "제조업"}},
        {"label": "Industry", "props": {"name": "자동차",    "sector": "제조업"}},
        {"label": "Industry", "props": {"name": "통신",      "sector": "서비스"}},
        {"label": "Industry", "props": {"name": "제조업",    "sector": "2차산업"}},
    ],
    "edges": list(corporate_data["edges"]) + [
        # 계열사 → 산업
        {"from": "삼성전자",   "rel": "BELONGS_TO_INDUSTRY", "to": "반도체"},
        {"from": "SK하이닉스", "rel": "BELONGS_TO_INDUSTRY", "to": "반도체"},
        {"from": "현대자동차", "rel": "BELONGS_TO_INDUSTRY", "to": "자동차"},
        {"from": "기아",       "rel": "BELONGS_TO_INDUSTRY", "to": "자동차"},
        {"from": "SK텔레콤",   "rel": "BELONGS_TO_INDUSTRY", "to": "통신"},
        # 경쟁사
        {"from": "삼성전자",   "rel": "COMPETES_WITH", "to": "SK하이닉스"},
        {"from": "현대자동차", "rel": "COMPETES_WITH", "to": "기아"},
    ],
}

reasoner_v2 = OntologyReasoner(corp_ontology, corporate_data_v2)

print("확장 후 그래프")
print(f"  노드 {len(corporate_data_v2['nodes'])}개, 엣지 {len(corporate_data_v2['edges'])}개")
print(f"  추가된 관계: BELONGS_TO_INDUSTRY(5), COMPETES_WITH(2)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# ① Naive RAG — 텍스트로 평탄화 후 TF-IDF 유사도 검색
# ─────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class NaiveRAG:
    """모든 그래프 정보를 자연어 문장으로 풀어쓴 뒤 TF-IDF 검색.
    → 클래스·관계의 의미는 잃고 단어 유사도에만 의존."""

    def __init__(self, data):
        self.chunks = []
        for n in data["nodes"]:
            p = n["props"]
            self.chunks.append(
                f"{n['label']} {p['name']} — " +
                ", ".join(f"{k}: {v}" for k, v in p.items() if k != "name")
            )
        for e in data["edges"]:
            self.chunks.append(f"{e['from']} {e['rel']} {e['to']}")
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
        self.mat = self.vec.fit_transform(self.chunks)

    def retrieve(self, question, top_k=5):
        q = self.vec.transform([question])
        sims = cosine_similarity(q, self.mat).ravel()
        top = sims.argsort()[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in top]

    def answer(self, question, top_k=5):
        hits = self.retrieve(question, top_k)
        return {
            "retrieved": hits,
            "context": "\n".join(f"- {c}" for c, _ in hits),
            "answer": (
                "[Naive 답변 시뮬레이션]\n"
                "검색된 청크에 인물·그룹명은 등장하지만, "
                "'1인 1회장' 같은 거버넌스 규칙은 텍스트에 표현되지 않아 "
                "위반 여부를 판단할 수 없습니다."
            ),
        }


naive = NaiveRAG(corporate_data_v2)

q = "이재용을 SK그룹의 회장으로 추가 임명해도 거버넌스 문제가 없나요?"
print("[Naive RAG] 검색 결과 Top-5")
for chunk, score in naive.retrieve(q, top_k=5):
    print(f"  ({score:.3f})  {chunk}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# ② Graph RAG — 시작 엔티티에서 N-hop BFS (스키마/제약 미사용)
# ─────────────────────────────────────────────────────────────

class GraphRAG:
    """노드·엣지는 알지만 클래스·도메인·치역·제약은 모르는 '평면 그래프' RAG.
    → 시작 노드에서 N-hop 트래버설로 이웃 트리플만 모음."""

    def __init__(self, reasoner: OntologyReasoner):
        self.reasoner = reasoner

    def retrieve(self, entities, max_hops=2):
        triples = []
        for e in entities:
            triples.extend(self.reasoner.find_related(e, max_hops=max_hops))
        return triples

    def answer(self, question, entities, max_hops=2):
        triples = self.retrieve(entities, max_hops)
        ctx = "\n".join(
            f"  ({t['from']}) --[{t['relation']}]--> ({t['to']})  (hop {t['depth']})"
            for t in triples
        )
        # HAS_CHAIRMAN 사실은 보지만 1인 1회장 규칙은 모름
        chair_facts = [t for t in triples if t["relation"] == "HAS_CHAIRMAN"]
        return {
            "triples": triples,
            "context": ctx,
            "chairman_facts": chair_facts,
            "answer": (
                f"[Graph 답변 시뮬레이션]\n"
                f"이재용 주변 {max_hops}-hop 그래프를 펼쳤습니다. "
                f"HAS_CHAIRMAN 사실 {len(chair_facts)}개 확인됨. "
                "그러나 '한 사람은 한 그룹의 회장만' 이라는 거버넌스 규칙은 "
                "그래프에 명시되어 있지 않아 자동 검증 불가."
            ),
        }


graph_rag = GraphRAG(reasoner_v2)

print("[Graph RAG] 이재용에서 2-hop 트래버설 (in-edge 포함)")
# in-edge 도 포함하도록 직접 펼치기
triples = []
for src, _, data in reasoner_v2.graph.in_edges("이재용", data=True):
    triples.append(f"  ({src}) --[{data['relation']}]--> (이재용)")
for _, tgt, data in reasoner_v2.graph.out_edges("이재용", data=True):
    triples.append(f"  (이재용) --[{data['relation']}]--> ({tgt})")
for t in triples:
    print(t)

print()
print("  → 이재용이 이미 삼성그룹 회장이라는 사실은 보임")
print("  → 그러나 SK그룹 회장으로 추가 임명이 '규칙 위반' 인지는 알 수 없음")
print("    (스키마의 1인 1회장 제약을 모르기 때문)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# ③ Ontology RAG — 스키마 + 추론 + 제약 검사
# ─────────────────────────────────────────────────────────────

class OntologyRAGLite:
    """LLM-free 비교 출력용. 스키마의 1인 1회장 제약을 알고 양방향 검색 + 제약 검사 수행."""

    def __init__(self, ontology, reasoner):
        self.ontology = ontology
        self.reasoner = reasoner

    def answer(self, question, person, target_group):
        # 1) 현재 회장 직책 (in-edge HAS_CHAIRMAN)
        current_chairman_of = []
        for src, _, d in self.reasoner.graph.in_edges(person, data=True):
            if d["relation"] == "HAS_CHAIRMAN":
                current_chairman_of.append(src)

        # 2) target_group 의 현 회장
        target_chair = None
        for _, tgt, d in self.reasoner.graph.out_edges(target_group, data=True):
            if d["relation"] == "HAS_CHAIRMAN":
                target_chair = tgt
                break

        # 3) 스키마 제약 위반 판정
        violations = []
        if current_chairman_of and target_group not in current_chairman_of:
            violations.append(
                f"1인 1회장 위반: {person} 은 이미 {current_chairman_of[0]} 의 회장"
            )
        if target_chair and target_chair != person:
            violations.append(
                f"Group HAS_CHAIRMAN cardinality(1) 위반: "
                f"{target_group} 은 이미 {target_chair} 이 회장"
            )

        verdict = "안전" if not violations else "위험 — 임명 거부"
        return {
            "person": person,
            "target_group": target_group,
            "current_chairman_of": current_chairman_of,
            "target_current_chairman": target_chair,
            "violations": violations,
            "verdict": verdict,
            "answer": (
                f"[Ontology 답변]\n"
                f"• 대상 인물: {person}\n"
                f"• 현재 회장직: {current_chairman_of or '없음'}\n"
                f"• 임명 대상 그룹: {target_group} (현 회장: {target_chair or '없음'})\n"
                f"• 위반: {violations or '없음'}\n"
                f"• 결론: {verdict}"
            ),
        }


onto_rag = OntologyRAGLite(corp_ontology, reasoner_v2)
result_onto = onto_rag.answer(q, person="이재용", target_group="SK그룹")
print(result_onto["answer"])


In [ ]:
# ─────────────────────────────────────────────────────────────
# ④ 세 방식을 같은 표로 비교
# ─────────────────────────────────────────────────────────────

result_naive = naive.answer(q, top_k=5)
result_graph = graph_rag.answer(q, entities=["이재용"], max_hops=2)
result_onto  = onto_rag.answer(q, person="이재용", target_group="SK그룹")


def section(title):
    print()
    print("─" * 78)
    print(f"  {title}")
    print("─" * 78)


section("질문")
print(f"  Q: {q}")

section("Naive RAG")
print("  검색 단위: 텍스트 청크 / 검색법: TF-IDF Top-5")
for c, s in result_naive["retrieved"]:
    print(f"    ({s:.2f}) {c}")
print(f"  → 답변: 거버넌스 규칙이 텍스트로 표현되지 않아 위반 판단 불가.")

section("Graph RAG")
print(f"  검색 단위: 트리플 / 검색법: 이재용에서 2-hop BFS")
print(f"  트리플 {len(result_graph['triples'])}개 수집")
print(f"  HAS_CHAIRMAN 사실 발견: {len(result_graph['chairman_facts'])}개")
print(f"  → 사실은 보이나 '1인 1회장' 규칙을 모르므로 자동 위반 판정 불가")

section("Ontology RAG")
print(f"  검색 단위: 스키마 + 트리플 + 제약 / 검색법: cardinality·1인1회장 직접 검증")
print(f"    {result_onto['person']} 현재 회장직: {result_onto['current_chairman_of']}")
print(f"    임명 대상: {result_onto['target_group']}  (현 회장: {result_onto['target_current_chairman']})")
print(f"    위반: {result_onto['violations']}")
print(f"  → 최종 판정: {result_onto['verdict']}")

section("요약 표")
print(f"  {'방식':<14}{'관계 인지':<12}{'규칙 인지':<12}{'제약 검증':<12}{'최종 답':<14}")
print(f"  {'-'*14}{'-'*12}{'-'*12}{'-'*12}{'-'*14}")
print(f"  {'Naive RAG':<14}{'X':<12}{'X':<12}{'X':<12}{'불명확':<14}")
print(f"  {'Graph RAG':<14}{'△':<12}{'X':<12}{'X':<12}{'불완전':<14}")
print(f"  {'Ontology RAG':<14}{'O':<12}{'O':<12}{'O':<12}{result_onto['verdict']:<14}")

section("핵심 인사이트")
print("  1. Naive RAG 는 텍스트 유사도만으로 '거버넌스 규칙' 을 표현하지 못한다.")
print("  2. Graph RAG 는 사실(이재용=삼성그룹 회장)은 보지만, 규칙(1인 1회장)은 모른다.")
print("  3. Ontology RAG 는 스키마의 cardinality 와 1인 1회장 제약을 알기 때문에")
print("     '추가 임명 위반' 을 자동으로 검출한다.")


---

## 5.7️⃣ 온톨로지 월드 모델 — 사실 ⊕ 규칙 = 추론 가능한 미니 세계

지금까지 만든 온톨로지는 *"검색 가능한 지식 그래프"* 였습니다.
여기에 **공리(axioms) + 제약(constraints) + 함의 규칙(entailment rules)** 을 더하면,
도메인을 **시뮬레이션·검증할 수 있는 월드 모델**이 됩니다.

### 월드 모델이 갖춰야 할 4가지 능력

| # | 능력 | 예시 (기업 거버넌스) |
|---|---|---|
| ① | **Entailment (함의 추론)** | `반도체 IS_A 제조업` ⊕ `삼성전자 BELONGS_TO_INDUSTRY 반도체` ⇒ `삼성전자 BELONGS_TO_INDUSTRY 제조업` |
| ② | **What-if (가설 추론)**     | "이재용을 SK그룹 회장으로 추가 임명하면 무슨 일이 벌어지나?" |
| ③ | **Consistency (일관성 검사)** | 새 사실이 1인 1회장 제약을 위반하는가? |
| ④ | **Action Simulation (상태 전이)** | 액션을 가상 적용 → OK 면 실제 상태 갱신 |

### LLM 월드 모델 vs 온톨로지 월드 모델

|  | LLM 월드 모델 | **온톨로지 월드 모델** |
|---|---|---|
| 추론 방식 | 통계적 (next-token 분포) | **형식 논리** (deduction) |
| 새 사실 생성 | 학습 데이터에서 외삽 | **공리에서 연역** |
| 정확성 보장 | 확률적 (환각 가능) | **결정론적 · 검증 가능** |
| 설명 가능성 | 낮음 | **추론 경로 제시 가능** |
| 약점 | 사실 검증 어려움 | 공리·제약 사람이 작성해야 함 |

> **베스트 프랙티스**: LLM = 자연어 인터페이스 + 후보 생성, 온톨로지 월드 모델 = 검증·추론 백엔드.
> 둘을 **결합**하면 OntologyRAG 가 됩니다.


In [ ]:
# ─────────────────────────────────────────────────────────────
# OntologyWorldModel — 사실 + 공리 + 제약 + 추론 엔진
# ─────────────────────────────────────────────────────────────
from copy import deepcopy


class OntologyWorldModel:
    """온톨로지를 \"추론 가능한 세계\"로 다루는 미니 시스템.

    - state         : 현재 사실(트리플) 집합
    - is_a_axioms   : 서브클래스 공리  (sub → super)
    - inverse_of    : 역관계 공리       (rel → inverse_rel)
    - constraints   : 제약 검사 함수 리스트
    """

    def __init__(self, ontology, data):
        self.ontology = ontology
        self.state = deepcopy(data)
        self.is_a_axioms: dict[str, str] = {}
        self.inverse_of:  dict[str, str] = {}
        self.constraints = [self._constraint_one_chairman_per_person,
                            self._constraint_group_cardinality_one]

    # ─── 공리 등록 ──────────────────────────────────
    def add_is_a(self, sub: str, sup: str):
        self.is_a_axioms[sub] = sup

    def add_inverse(self, rel: str, inv: str):
        self.inverse_of[rel] = inv
        self.inverse_of[inv] = rel

    # ─── ① Entailment ───────────────────────────────
    def deduce(self, state=None) -> list[dict]:
        """공리에서 함의된 새 트리플 생성. IS_A 전이성 + 역관계."""
        s = state if state is not None else self.state
        existing = {(e["from"], e["rel"], e["to"]) for e in s["edges"]}
        derived = []

        # IS_A 전이: (X, BELONGS_TO_INDUSTRY, 반도체) ⊕ (반도체 IS_A 제조업)
        #         ⇒ (X, BELONGS_TO_INDUSTRY, 제조업)
        for edge in list(s["edges"]):
            obj = edge["to"]
            if obj in self.is_a_axioms:
                sup = self.is_a_axioms[obj]
                new = {"from": edge["from"], "rel": edge["rel"], "to": sup}
                key = (new["from"], new["rel"], new["to"])
                if key not in existing:
                    derived.append({**new, "_via": f"IS_A({obj}→{sup})"})
                    existing.add(key)

        # 역관계: (A, REL, B) ⇒ (B, INV_REL, A)
        for edge in list(s["edges"]):
            if edge["rel"] in self.inverse_of:
                inv = self.inverse_of[edge["rel"]]
                new = {"from": edge["to"], "rel": inv, "to": edge["from"]}
                key = (new["from"], new["rel"], new["to"])
                if key not in existing:
                    derived.append({**new, "_via": f"INV({edge['rel']})"})
                    existing.add(key)
        return derived

    # ─── ② What-if ──────────────────────────────────
    def hypothetical(self, add_edges: list[dict]) -> dict:
        ghost = deepcopy(self.state)
        ghost["edges"].extend(add_edges)
        new_derived = self.deduce(ghost)
        violations = self.check_consistency(ghost)
        return {
            "added": add_edges, "derived": new_derived,
            "violations": violations, "valid": len(violations) == 0,
        }

    # ─── ③ Consistency ──────────────────────────────
    def check_consistency(self, state=None) -> list[str]:
        s = state if state is not None else self.state
        violations = []
        for c in self.constraints:
            violations.extend(c(s))
        return violations

    def _constraint_one_chairman_per_person(self, s) -> list[str]:
        """제약: 한 사람은 최대 1개의 Group 의 회장."""
        chair_of = {}  # person → [groups]
        for e in s["edges"]:
            if e["rel"] == "HAS_CHAIRMAN":
                chair_of.setdefault(e["to"], []).append(e["from"])
        return [
            f"⚠ {p} 는 {gs} 의 회장 ({len(gs)}개) — 1인 1회장 위반"
            for p, gs in chair_of.items() if len(gs) > 1
        ]

    def _constraint_group_cardinality_one(self, s) -> list[str]:
        """제약: 한 Group 은 정확히 1명의 회장."""
        group_chairs = {}  # group → [persons]
        for e in s["edges"]:
            if e["rel"] == "HAS_CHAIRMAN":
                group_chairs.setdefault(e["from"], []).append(e["to"])
        return [
            f"⚠ {g} 에 회장 {len(ps)}명 — Group HAS_CHAIRMAN cardinality(1) 위반"
            for g, ps in group_chairs.items() if len(ps) > 1
        ]

    # ─── ④ Action Simulation ────────────────────────
    def simulate_action(self, action: dict) -> dict:
        result = self.hypothetical(action["add"])
        if result["valid"]:
            self.state["edges"].extend(action["add"])
            result["committed"] = True
        else:
            result["committed"] = False
        return result


# ── 월드 초기화 (corporate_data_v2 기반) ──────────────
world = OntologyWorldModel(corp_ontology, corporate_data_v2)

# IS_A 공리 등록: 산업 분류 계층
world.add_is_a("반도체",   "제조업")
world.add_is_a("자동차",   "제조업")
world.add_is_a("제조업",   "산업")
world.add_is_a("통신",     "서비스")

# 역관계 공리
world.add_inverse("HAS_CHAIRMAN", "CHAIRMAN_OF")
world.add_inverse("HAS_CEO",      "CEO_OF")

print("월드 초기화 완료")
print(f"  사실: {len(world.state['edges'])}개")
print(f"  IS_A 공리: {world.is_a_axioms}")
print(f"  역관계 공리: {world.inverse_of}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 월드 모델의 4가지 능력 시연
# ─────────────────────────────────────────────────────────────

def section(title):
    print(); print("─" * 78); print(f"  {title}"); print("─" * 78)


# ════ ① Entailment ════
section("① Entailment — 사실 + IS_A·역관계 공리 → 새 사실 자동 도출")
derived = world.deduce()
print(f"  명시 사실 {len(world.state['edges'])}개 → 추가로 도출된 사실 {len(derived)}개")
for d in derived[:12]:
    print(f"    + ({d['from']}) --[{d['rel']}]--> ({d['to']})    via {d['_via']}")
if len(derived) > 12:
    print(f"    ... 외 {len(derived)-12}개")
print()
print("  → \"삼성전자 BELONGS_TO_INDUSTRY 제조업\" 같은 사실은 명시한 적 없지만,")
print("    \"반도체 IS_A 제조업\" 공리에서 자동 도출됨 (연역적 closure)")


# ════ ② What-if ════
section("② What-if — '이재용을 SK그룹 회장으로 추가 임명' 가상 시나리오")
whatif = world.hypothetical([
    {"from": "SK그룹", "rel": "HAS_CHAIRMAN", "to": "이재용"}
])
print(f"  가상 추가: {whatif['added']}")
print(f"  새 함의: {len(whatif['derived'])}개")
print(f"  제약 위반: {len(whatif['violations'])}개")
for v in whatif["violations"]:
    print(f"    {v}")
print(f"  → 가상 세계 유효? {whatif['valid']}")


# ════ ③ Action ════
section("③ Action Simulation — 위 액션 실제 적용 시도")
result = world.simulate_action({
    "add": [{"from": "SK그룹", "rel": "HAS_CHAIRMAN", "to": "이재용"}]
})
print(f"  commit 여부: {result['committed']}")
if not result["committed"]:
    print(f"  사유: 제약 위반 — {result['violations'][0]}")
print(f"  → 월드 상태 변경되지 않음. 현재 사실 수: {len(world.state['edges'])}")


# ════ ④ Alternative ════
section("④ 대안 액션 — '정의선 → 삼성SDS 사외이사' 같은 안전한 변경")
# 회장 임명이 아니라 다른 역할 (HAS_CEO 가 아닌 그냥 추가 관계)
result2 = world.simulate_action({
    "add": [{"from": "삼성SDS", "rel": "COMPETES_WITH", "to": "SK텔레콤"}]
})
print(f"  commit 여부: {result2['committed']}")
print(f"  위반: {result2['violations'] or '없음'}")
print(f"  → 월드 상태 갱신됨. 현재 사실 수: {len(world.state['edges'])}")


# ════ 핵심 인사이트 ════
section("💡 핵심 인사이트")
print("  • Entailment   → IS_A 공리로 산업 계층을 자동 정리 (\"반도체→제조업→산업\")")
print("  • What-if      → 이재용 추가 임명의 효과를 실제 변경 없이 미리 확인")
print("  • Consistency  → 1인 1회장 / HAS_CHAIRMAN cardinality 결정적 차단")
print("  • Simulation   → 위반 액션은 거부, 안전한 액션만 commit (Plan & Act)")
print()
print("  → 이 4가지가 합쳐지면 단순 \"질의 응답\" 을 넘어,")
print("    LLM Agent 의 \"안전한 행동 모델 / 자기 검증 백엔드\" 가 됩니다.")


---

## 6️⃣ 실전 응용: 도메인별 온톨로지 RAG 활용 사례

### 기업 거버넌스 도메인 (이번 실습 도메인)

```
활용 사례:
  - 회장·CEO 임명 시 1인 1회장 등 거버넌스 규칙 자동 검증
  - 순환출자 / 상호출자 탐지
  - 계열사 분류·산업 계층 자동 추론 (IS_A)
  - ESG·공시 자료 자동 분류 및 일관성 검사

온톨로지 예: FIBO (Financial Industry Business Ontology), 공정거래위원회 기업집단 데이터
```

### 법률 도메인

```
활용 사례:
  - 법률 조항 간 관계 분석
  - 판례 기반 추론 (유사 판례 검색)
  - 계약서 자동 검토 (조항 충돌 탐지)
  - 규정 준수 확인

온톨로지 클래스: 법률, 조항, 판례, 법원, 당사자, 판결
핵심 관계: 인용(cites), 개정(amends), 적용(appliesTo)
```

### 금융 도메인

```
활용 사례:
  - 기업 관계 분석 (지분, 계열사)
  - 사기 탐지 (이상 거래 패턴)
  - 리스크 전파 분석
  - 규제 영향 분석

온톨로지 클래스: 기업, 주식, 펀드, 규제, 거래, 계좌
핵심 관계: 소유(owns), 거래(trades), 보증(guarantees)
```


In [ ]:
# 법률 도메인 온톨로지 예시

legal_ontology = OntologySchema("법률 온톨로지")

legal_ontology.add_class("Law", properties={"name": "string", "enacted_date": "date", "status": "string"})
legal_ontology.add_class("Article", properties={"number": "string", "content": "string"})
legal_ontology.add_class("Case", properties={"case_number": "string", "date": "date", "court": "string"})
legal_ontology.add_class("Party", properties={"name": "string", "role": "string"})
legal_ontology.add_class("Verdict", properties={"result": "string", "summary": "string"})

legal_ontology.add_relation("contains", "Law", "Article", "one-to-many")
legal_ontology.add_relation("cites", "Case", "Article", "many-to-many")
legal_ontology.add_relation("involvedIn", "Party", "Case", "many-to-many")
legal_ontology.add_relation("hasVerdict", "Case", "Verdict", "one-to-one")
legal_ontology.add_relation("amends", "Law", "Law", "many-to-many")
legal_ontology.add_relation("precedentOf", "Case", "Case", "many-to-many")

legal_ontology.add_constraint("Case는 반드시 하나의 Verdict를 가져야 한다")
legal_ontology.add_constraint("amends 관계에서 개정법은 반드시 원법보다 나중에 제정되어야 한다")

legal_ontology.display()

In [ ]:
# 금융 도메인 온톨로지 예시

finance_ontology = OntologySchema("금융 온톨로지")

finance_ontology.add_class("Corporation", properties={"name": "string", "ticker": "string", "sector": "string"})
finance_ontology.add_class("Stock", properties={"ticker": "string", "price": "float", "market_cap": "float"})
finance_ontology.add_class("Fund", properties={"name": "string", "type": "string", "aum": "float"})
finance_ontology.add_class("Regulation", properties={"name": "string", "authority": "string"})
finance_ontology.add_class("Transaction", properties={"id": "string", "amount": "float", "date": "date"})

finance_ontology.add_relation("owns", "Corporation", "Corporation", "many-to-many")
finance_ontology.add_relation("issues", "Corporation", "Stock", "one-to-one")
finance_ontology.add_relation("holdsPosition", "Fund", "Stock", "many-to-many")
finance_ontology.add_relation("regulatedBy", "Corporation", "Regulation", "many-to-many")
finance_ontology.add_relation("executes", "Corporation", "Transaction", "one-to-many")
finance_ontology.add_relation("subsidiaryOf", "Corporation", "Corporation", "many-to-one")

finance_ontology.add_constraint("subsidiaryOf 관계에서 순환 참조는 허용되지 않는다")
finance_ontology.add_constraint("owns 비율의 합은 100%를 초과할 수 없다")

finance_ontology.display()

print("\n" + "=" * 60)
print("도메인별 온톨로지 설계 완료")
print("  - 기업 거버넌스: 그룹-계열사-임원-학교 + 1인 1회장 제약")
print("  - 법률: 법률-조항-판례-판결 관계 모델링")
print("  - 금융: 기업-주식-펀드-규제 관계 모델링")


---

## 정리

### 핵심 요약

| 항목 | 그래프 RAG | 온톨로지 RAG |
|------|------------|-------------|
| 스키마 | 암묵적 (데이터 주도) | 명시적 (도메인 전문가 설계) |
| 추론 | 경로 탐색 | 논리적 추론 (상속, 대칭, 전이) |
| 검증 | 제한적 | 제약조건 기반 검증 |
| 구축 비용 | 중간 | 높음 (도메인 전문성 필요) |
| 적합 도메인 | 일반 관계 질문 | 전문 도메인 (기업 거버넌스, 법률, 금융) |

### 실무 적용 가이드

1. **단순 QA** → 벡터 RAG로 충분
2. **관계/경로 질문** → 그래프 RAG 추천
3. **도메인 전문 추론** → 온톨로지 RAG 필요
4. **최적 방안** → 하이브리드 (벡터 + 그래프 + 온톨로지)


In [ ]:
print("=" * 60)
print("온톨로지 RAG 실습 완료!")
print("=" * 60)
print()
print("학습 내용 정리:")
print("  1. 온톨로지 개념: 클래스, 프로퍼티, 관계, 제약조건")
print("  2. 벡터 RAG vs 그래프 RAG vs 온톨로지 RAG 비교")
print("  3. Apache AGE: PostgreSQL 기반 그래프 DB")
print("  4. Cypher 쿼리 기초 및 AGE 사용법")
print("  5. 온톨로지 RAG 파이프라인 구현 (기업 거버넌스 도메인)")
print("  6. 도메인별 활용 사례: 기업 거버넌스, 법률, 금융")
